# ARC Prize 2026 (ARC-AGI-2) — Notebook de entrega

Corre **offline** (sin internet). Pipeline: cargar test → (opcional) Test-Time Training →
inferencia AIRV (augment + voto) → `submission.json`.

**Antes de enviar**, adjunta como *Datasets* de este notebook:
1. La competencia `arc-prize-2026-arc-agi-2` (datos).
2. El código de este repo (sube `src/` como dataset, p.ej. `arc-agi2-code`).
3. Los pesos del modelo base + LoRA entrenado (p.ej. `arc-agi2-model`).

Si el modelo no está disponible, el notebook cae a una baseline y **siempre** escribe un submission válido.

In [ ]:
# === Configuración ===
import os, sys, glob, json
from pathlib import Path

# Datos de la competencia
COMP_DIR = '/kaggle/input/arc-prize-2026-arc-agi-2'
if not os.path.isdir(COMP_DIR):
    COMP_DIR = str(Path.cwd().parent / 'data')  # fallback local
TEST_CHALLENGES = os.path.join(COMP_DIR, 'arc-agi_test_challenges.json')

# Código del repo (busca un dataset que contenga src/arc)
CODE_DIR = None
for c in glob.glob('/kaggle/input/*/src') + ['../src', 'src']:
    if os.path.isdir(os.path.join(c, 'arc')):
        CODE_DIR = c; break
assert CODE_DIR, 'No se encontró src/arc. Adjunta el código como dataset.'
sys.path.insert(0, CODE_DIR)

# Modelo (base + adaptador LoRA). Ajusta a tu dataset.
MODEL_DIR = None
for c in glob.glob('/kaggle/input/*'):
    if glob.glob(os.path.join(c, '**', 'config.json'), recursive=True):
        MODEL_DIR = c; break
LORA_DIR = None
for c in glob.glob('/kaggle/input/*'):
    if glob.glob(os.path.join(c, '**', 'adapter_config.json'), recursive=True):
        LORA_DIR = os.path.dirname(glob.glob(os.path.join(c, '**', 'adapter_config.json'), recursive=True)[0]); break

ENABLE_TTT = False   # activa test-time training (más lento; requiere tuning)
N_AUG = 8            # nº de vistas augmentadas para AIRV
MAX_NEW_TOKENS = 2048
OUT_PATH = 'submission.json'
print('COMP_DIR =', COMP_DIR); print('CODE_DIR =', CODE_DIR)
print('MODEL_DIR =', MODEL_DIR); print('LORA_DIR =', LORA_DIR)

In [ ]:
# === Cargar tareas de test ===
from arc.data import load_dataset
tasks = load_dataset(TEST_CHALLENGES)
print('Tareas de test:', len(tasks))

In [ ]:
# === Cargar modelo (con fallback a baseline si no está disponible) ===
USE_MODEL = MODEL_DIR is not None
model = tok = None
if USE_MODEL:
    try:
        from arc.model import load_model, generate_grid
        import torch
        model, tok = load_model(MODEL_DIR, load_4bit=False, dtype='bfloat16',
                                device_map='cuda' if torch.cuda.is_available() else 'cpu')
        if LORA_DIR:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, LORA_DIR)
        model.eval()
        print('Modelo cargado.')
    except Exception as e:
        print('Fallo al cargar el modelo, usando baseline:', e)
        USE_MODEL = False

In [ ]:
# === (Opcional) Test-Time Training por tarea ===
# Adapta un LoRA ligero sobre los pares 'train' (augmentados) de CADA tarea antes de predecir.
# Implementación de referencia; ajustar pasos/lr/batch al presupuesto de runtime.
def ttt_adapt(base_model, tok, task, steps=20, lr=1e-4):
    import torch
    from arc.augment import augment_task, color_permutation
    from arc.grid import format_task_prompt, grid_to_text
    samples = []
    for a in range(4):
        d4, cmap = (0, list(range(10))) if a == 0 else (a % 8, color_permutation(a))
        at = augment_task(task, d4, cmap)
        for i in range(len(at.train)):
            ctx = at.train[:i] + at.train[i+1:]
            if not ctx: continue
            p = format_task_prompt(ctx, at.train[i].input)
            c = '\n' + grid_to_text(at.train[i].output) + '\n'
            samples.append((p, c))
    if not samples: return base_model
    opt = torch.optim.AdamW([q for q in base_model.parameters() if q.requires_grad], lr=lr)
    base_model.train()
    for s in range(steps):
        p, c = samples[s % len(samples)]
        enc = tok(p + c, return_tensors='pt').to(base_model.device)
        out = base_model(**enc, labels=enc['input_ids'])
        out.loss.backward(); opt.step(); opt.zero_grad()
    base_model.eval()
    return base_model

In [ ]:
# === Inferencia AIRV + escritura de submission ===
from arc.infer import predict_task, two_attempts
from collections import Counter
from arc.grid import grid_to_text

def baseline_predict(task):
    outs = [grid_to_text(p.output) for p in task.train if p.output]
    common = Counter(outs).most_common(1)
    from arc.grid import text_to_grid
    mc = text_to_grid(common[0][0]) if common else [[0]]
    return [{'attempt_1': tp.input, 'attempt_2': mc} for tp in task.test]

submission = {}
for n, (tid, task) in enumerate(tasks.items()):
    try:
        if USE_MODEL:
            if ENABLE_TTT:
                ttt_adapt(model, tok, task)
            predict_fn = lambda prompt: generate_grid(model, tok, prompt,
                                                       max_new_tokens=MAX_NEW_TOKENS,
                                                       constrained=True)
            submission[tid] = predict_task(task, predict_fn, n_aug=N_AUG)
        else:
            submission[tid] = baseline_predict(task)
    except Exception as e:
        print(f'[{tid}] error -> baseline:', e)
        submission[tid] = baseline_predict(task)
    if (n+1) % 20 == 0:
        print(f'{n+1}/{len(tasks)} tareas')

with open(OUT_PATH, 'w') as f:
    json.dump(submission, f)
print('Escrito', OUT_PATH, 'con', len(submission), 'tareas')

In [ ]:
# === Validación de formato ===
sub = json.load(open(OUT_PATH))
assert all(isinstance(v, list) for v in sub.values())
for tid, task in tasks.items():
    assert len(sub[tid]) == len(task.test), tid
    for att in sub[tid]:
        assert 'attempt_1' in att and 'attempt_2' in att
print('Formato OK:', len(sub), 'tareas')